## Check Colab Connection and GPU

In [1]:
import os
import sys

# 1. Check if we are in a Colab environment
in_colab = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f"Running in Google Colab: {in_colab}")

# 2. Check the Python executable path (Colab usually uses /usr/bin/python3)
print(f"Python execution path: {sys.executable}")

# 3. Check the GPU attached to this kernel
gpu_info = os.popen("nvidia-smi -L 2>/dev/null").read().strip()
if gpu_info:
    print("GPU detected:")
    print(gpu_info)
else:
    print("No GPU attached to this runtime. Enable GPU in Runtime settings.")
!nvidia-smi -L

Running in Google Colab: True
Python execution path: /usr/bin/python3
GPU detected:
GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-9edf3735-2437-9ab6-a323-ea727cc030ac)
GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-9edf3735-2437-9ab6-a323-ea727cc030ac)


### Installation

In [2]:
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm synthetic-data-kit==0.0.3
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install synthetic-data-kit==0.0.3
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [3]:
%%capture
#@title Colab Extra Install { display-mode: "form" }
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    # Use Unsloth's official auto-installer for optimal compilation on high-end GPUs like H100
    !wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -
    !uv pip install -qqq torchvision bitsandbytes xformers vllm

!uv pip install -qqq transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

In [4]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
    # Qwen3 new models
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    # Other very popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.3-70B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory (~35GB on H100)
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.5: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00009.safetensors:   0%|          | 0.00/4.89G [00:00<?, ?B/s]

model-00002-of-00009.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00009.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

## Mount Google Drive & Setup Paths
Upload the required files to `My Drive/irp_retraining/` before running.

## Mount Google Drive & Setup Paths
Upload the required files to `My Drive/irp_retraining/` before running.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/irp_retraining")
ASSET_AWARE_DS = DRIVE_ROOT / "datasets" / "ir_playbooks_alpaca_with_assets.jsonl"
MITRE_MAPPING = DRIVE_ROOT / "helper_scripts" / "mitre_incident_mapping.json"
ASSET_PROFILES = DRIVE_ROOT / "helper_scripts" / "asset_profiles.py"

OUTPUT_REWRITTEN = DRIVE_ROOT / "datasets" / "ir_playbooks_alpaca_rewritten.jsonl"
OUTPUT_SYNTHETIC = DRIVE_ROOT / "datasets" / "ir_playbooks_synthetic_v2.jsonl"

# Verify files exist
for p in [ASSET_AWARE_DS, MITRE_MAPPING, ASSET_PROFILES]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {p}")


In [ ]:
# Load the asset-aware dataset
with open(ASSET_AWARE_DS, "r", encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f if line.strip()]

asset_aware_rows = [(i, r) for i, r in enumerate(all_rows) if r.get("_asset_aware")]
original_rows = [(i, r) for i, r in enumerate(all_rows) if not r.get("_asset_aware")]

print(f"Total rows: {len(all_rows)}")
print(f"  Asset-aware (need output rewriting): {len(asset_aware_rows)}")
print(f"  Original format (keep as-is): {len(original_rows)}")

# Load MITRE mapping
with open(MITRE_MAPPING, "r", encoding="utf-8") as f:
    mitre_data = json.load(f)
print(f"\nMITRE campaigns loaded: {len(mitre_data['campaigns'])}")


## Job 1: Output Rewriting (Asset-Aware)

For each of the 110 asset-aware rows, use the 70B model to rewrite the `output` field
so it references specific hostnames, IPs, and tool names from the injected asset context.
This teaches the model to produce asset-specific playbooks.

In [ ]:
import torch

REWRITE_SYSTEM = (
    "You are an incident response expert. You rewrite IR playbooks to reference "
    "specific organization assets. You preserve the exact format and all 5 NIST phases."
)

REWRITE_USER_TEMPLATE = """Below is an IR playbook and the organization's asset inventory.
Rewrite the playbook so that:
1. Generic tool names (EDR, SIEM, Firewall, etc.) are replaced with specific asset names, IPs, and hostnames from the inventory
2. Action descriptions reference specific assets by hostname and IP where relevant
3. Add an "Affected Assets:" section BEFORE "Final status:" listing each referenced asset and its role in the response
4. Keep ALL 5 NIST phases (Identification, Containment, Eradication, Recovery, Lessons Learned)
5. Keep the EXACT format: Phase/Action/Tools/Target response time per phase, Final status, Total response duration
6. Do NOT add any text before "Playbook:" or after "Total response duration"

{asset_context}

Original playbook:
{original_output}

Rewrite the playbook now. Output ONLY the rewritten playbook starting with "Playbook:" and nothing else.
"""


def rewrite_output(original_output, asset_context, max_new_tokens=1024, temperature=0.3):
    """Use the 70B model to rewrite a playbook output to reference specific assets."""
    user_content = REWRITE_USER_TEMPLATE.format(
        asset_context=asset_context,
        original_output=original_output
    )
    messages = [
        {"role": "system", "content": REWRITE_SYSTEM},
        {"role": "user", "content": user_content},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inputs.to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
        )
    generated = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    
    # Ensure it starts with "Playbook:"
    if "Playbook:" in generated:
        generated = generated[generated.index("Playbook:"):]
    
    return generated


def extract_asset_context(input_text):
    """Extract the Organization assets: block from the input field."""
    marker = "Organization assets:"
    idx = input_text.find(marker)
    if idx == -1:
        return None
    return input_text[idx:]


def validate_rewrite(text):
    """Basic validation that rewrite has the right structure."""
    required = ["Phase: Identification", "Phase: Containment", "Phase: Eradication",
                "Phase: Recovery", "Phase: Lessons Learned", "Affected Assets:",
                "Final status:"]
    return all(phrase in text for phrase in required)

print("Rewrite functions ready.")


In [ ]:
# Run output rewriting on all asset-aware rows
rewritten_rows = list(all_rows)  # copy
success = 0
failed_indices = []

for batch_num, (idx, row) in enumerate(asset_aware_rows):
    asset_context = extract_asset_context(row["input"])
    if asset_context is None:
        print(f"  [SKIP] Row {idx}: no asset context found")
        failed_indices.append(idx)
        continue

    original_output = row["output"]
    
    # Try up to 2 attempts
    rewritten = None
    for attempt in range(2):
        candidate = rewrite_output(original_output, asset_context)
        if validate_rewrite(candidate):
            rewritten = candidate
            break
    
    if rewritten:
        rewritten_rows[idx] = dict(row)
        rewritten_rows[idx]["output"] = rewritten
        success += 1
    else:
        failed_indices.append(idx)
    
    if (batch_num + 1) % 10 == 0:
        print(f"  Rewritten {batch_num + 1}/{len(asset_aware_rows)} (success: {success})")
    
    # Save checkpoint every 25 rows
    if (batch_num + 1) % 25 == 0:
        with open(OUTPUT_REWRITTEN, "w", encoding="utf-8") as f:
            for r in rewritten_rows:
                clean = {k: v for k, v in r.items() if not k.startswith("_")}
                f.write(json.dumps(clean) + "\n")
        print(f"  [Checkpoint saved to {OUTPUT_REWRITTEN}]")

print(f"\nOutput rewriting complete: {success}/{len(asset_aware_rows)} successful")
if failed_indices:
    print(f"Failed rows (kept original output): {failed_indices}")

# Final save
with open(OUTPUT_REWRITTEN, "w", encoding="utf-8") as f:
    for r in rewritten_rows:
        clean = {k: v for k, v in r.items() if not k.startswith("_")}
        f.write(json.dumps(clean) + "\n")
print(f"Saved to {OUTPUT_REWRITTEN}")


## Job 2: Synthetic Playbook Generation (MITRE-Seeded)

Generate 50-80 new Alpaca-format playbooks using MITRE ATT&CK campaigns as seeds.
Each campaign is used to generate 3-4 variations.

In [11]:
SYNTH_SYSTEM = (
    "You are an incident response expert. Generate detailed incident response playbooks "
    "in the exact format specified. Include all 5 NIST phases. Use real security tool names. "
    "Be specific and actionable."
)

SYNTH_USER_TEMPLATE = """Generate an incident response playbook for this real-world attack scenario.

Attack context:
- Campaign: {campaign_name}
- Incident type: {incident_type}
- MITRE ATT&CK tactics: {tactics}
- Target asset: {target_asset}
- Severity: {severity}
- Background: {notes}

Generate the output in this EXACT format (no JSON, just plain text):

Playbook:

Phase: Identification
Action: [specific actions for this incident]
Tools: [real tool names like CrowdStrike Falcon, Splunk, Palo Alto, etc.]
Target response time (minutes): [number]

Phase: Containment
Action: [specific containment actions]
Tools: [real tool names]
Target response time (minutes): [number]

Phase: Eradication
Action: [specific eradication actions]
Tools: [real tool names]
Target response time (minutes): [number]

Phase: Recovery
Action: [specific recovery actions]
Tools: [real tool names]
Target response time (minutes): [number]

Phase: Lessons Learned
Action: [specific post-incident actions]
Tools: [real tool names]
Target response time (minutes): [number]

Final status: Resolved
Total response duration (minutes): [sum of all phases]

Output ONLY the playbook text above. No explanations, no JSON, no markdown.
"""


def generate_playbook(campaign, max_new_tokens=768, temperature=0.7):
    """Generate a single synthetic playbook from a MITRE campaign seed."""
    # Handle both string and object-style tactics
    def fmt_tactic(t):
        if isinstance(t, dict):
            tid = f" ({t['technique_id']})" if t.get("technique_id") else ""
            return f"{t['tactic']}: {t['technique']}{tid}"
        return str(t)
    tactics_str = "; ".join(fmt_tactic(t) for t in campaign["tactics"])
    user_content = SYNTH_USER_TEMPLATE.format(
        campaign_name=campaign["name"],
        incident_type=campaign["incident_type"],
        tactics=tactics_str,
        target_asset=campaign["target_asset"],
        severity=campaign["severity"],
        notes=campaign.get("notes", f"Attributed to {(campaign.get('attributed_group') or {}).get('name', 'Unknown')} ({campaign.get('attack_id', '')}). Source: {campaign.get('source_url', '')}"),
    )
    messages = [
        {"role": "system", "content": SYNTH_SYSTEM},
        {"role": "user", "content": user_content},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inputs.to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
        )
    generated = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    
    if "Playbook:" in generated:
        generated = generated[generated.index("Playbook:"):]
    
    return generated


def validate_synthetic(output_text):
    """Validate that a synthetic playbook has all required phases."""
    required_phases = ["Phase: Identification", "Phase: Containment", "Phase: Eradication",
                       "Phase: Recovery", "Phase: Lessons Learned"]
    has_phases = all(p in output_text for p in required_phases)
    has_final = "Final status:" in output_text
    # Check for real tool names (at least 2)
    real_tools = ["CrowdStrike", "Splunk", "Palo Alto", "Veeam", "Suricata", "Darktrace",
                  "Carbon Black", "Velociraptor", "YARA", "Wireshark", "Nessus", "OpenVAS",
                  "Snort", "Cloudflare", "Okta", "Duo", "CyberArk", "MISP", "Volatility",
                  "Burp Suite", "Tenable", "Qualys", "SentinelOne", "Microsoft Defender",
                  "Wazuh", "ELK", "Elastic", "Nagios", "Zabbix"]
    tool_count = sum(1 for t in real_tools if t.lower() in output_text.lower())
    return has_phases and has_final and tool_count >= 2


def build_alpaca_row(campaign, output_text):
    """Convert a synthetic playbook into Alpaca format."""
    # Format tactics: handle both string and object-style
    def fmt_line(t):
        if isinstance(t, dict):
            tid = f" ({t['technique_id']})" if t.get("technique_id") else ""
            return f"- {t['tactic']}: {t['technique']}{tid}"
        return f"- {t}"
    tactics_lines = "\n".join(fmt_line(t) for t in campaign["tactics"])
    
    # Extract initial vector from first tactic
    first = campaign["tactics"][0]
    if isinstance(first, dict):
        initial_vector = first["technique"]
    else:
        initial_vector = first.split(":")[1].strip().split("(")[0].strip() if ":" in first else first
    
    tags = campaign["incident_type"].lower().replace(" ", "_")
    
    input_text = (
        f"Incident type: {campaign['incident_type']}\n"
        f"Target asset: {campaign['target_asset']}\n"
        f"Detection source: MITRE ATT&CK Campaign - {campaign['name']}\n"
        f"Initial vector: {initial_vector}\n"
        f"Tactics and techniques:\n{tactics_lines}\n"
        f"Severity: {campaign['severity']}\n"
        f"Tags: {tags}, mitre_attack"
    )
    
    return {
        "instruction": "Create a detailed incident response playbook for the following cyber incident.",
        "input": input_text,
        "output": output_text,
    }

print("Synthetic generation functions ready.")


Synthetic generation functions ready.


In [12]:
import random
random.seed(42)

campaigns = mitre_data["campaigns"]
VARIATIONS_PER_CAMPAIGN = 3  # 21 campaigns x 3 = 63 target rows
TARGET_COUNT = len(campaigns) * VARIATIONS_PER_CAMPAIGN
MAX_ATTEMPTS = TARGET_COUNT * 3

synthetic_rows = []
campaign_idx = 0
variation = 0
attempts = 0

print(f"Generating ~{TARGET_COUNT} synthetic playbooks from {len(campaigns)} MITRE campaigns...")
print(f"({VARIATIONS_PER_CAMPAIGN} variations each)\n")

while len(synthetic_rows) < TARGET_COUNT and attempts < MAX_ATTEMPTS:
    campaign = campaigns[campaign_idx]
    
    # Vary temperature slightly for diversity
    temp = 0.6 + (variation * 0.15)  # 0.6, 0.75, 0.9
    output_text = generate_playbook(campaign, temperature=min(temp, 0.95))
    attempts += 1
    
    if validate_synthetic(output_text):
        row = build_alpaca_row(campaign, output_text)
        synthetic_rows.append(row)
        variation += 1
        
        if variation >= VARIATIONS_PER_CAMPAIGN:
            variation = 0
            campaign_idx += 1
            if campaign_idx >= len(campaigns):
                break
        
        if len(synthetic_rows) % 10 == 0:
            print(f"  Generated {len(synthetic_rows)}/{TARGET_COUNT}")
    else:
        print(f"  [REJECT] Campaign: {campaign['name']} (attempt {attempts})")

print(f"\nSynthetic generation complete: {len(synthetic_rows)} rows")
print(f"  Attempts: {attempts}, Rejection rate: {(attempts - len(synthetic_rows))/max(attempts,1):.1%}")

# Save synthetic rows
with open(OUTPUT_SYNTHETIC, "w", encoding="utf-8") as f:
    for r in synthetic_rows:
        f.write(json.dumps(r) + "\n")
print(f"Saved to {OUTPUT_SYNTHETIC}")


Generating ~57 synthetic playbooks from 19 MITRE campaigns...
(3 variations each)

  Generated 10/57
  Generated 20/57
  Generated 30/57
  Generated 40/57
  Generated 50/57

Synthetic generation complete: 57 rows
  Attempts: 57, Rejection rate: 0.0%
Saved to /content/drive/MyDrive/irp_retraining/datasets/ir_playbooks_synthetic_v2.jsonl


## Summary & Validation

In [13]:
from collections import Counter

# Summarize rewritten dataset
with open(OUTPUT_REWRITTEN, "r", encoding="utf-8") as f:
    rewritten = [json.loads(l) for l in f if l.strip()]

print(f"=== Rewritten Base Dataset ===")
print(f"Total rows: {len(rewritten)}")
has_affected = sum(1 for r in rewritten if "Affected Assets:" in r.get("output", ""))
print(f"Rows with Affected Assets section: {has_affected}")

# Summarize synthetic dataset
with open(OUTPUT_SYNTHETIC, "r", encoding="utf-8") as f:
    synthetic = [json.loads(l) for l in f if l.strip()]

print(f"\n=== Synthetic Dataset ===")
print(f"Total rows: {len(synthetic)}")

synth_types = Counter()
for r in synthetic:
    for line in r.get("input", "").split("\n"):
        if line.startswith("Incident type:"):
            synth_types[line.replace("Incident type:", "").strip()] += 1
            break

print("Synthetic incident type distribution:")
for itype, count in synth_types.most_common():
    print(f"  {itype}: {count}")

print(f"\n=== Files saved to Google Drive ===")
print(f"  {OUTPUT_REWRITTEN}")
print(f"  {OUTPUT_SYNTHETIC}")
print(f"\nDownload these files and place in your local project datasets/ folder.")


=== Rewritten Base Dataset ===
Total rows: 137
Rows with Affected Assets section: 110

=== Synthetic Dataset ===
Total rows: 57
Synthetic incident type distribution:
  Ransomware: 3
  Data Breach: 3
  Phishing: 3
  Malware Infection: 3
  DDoS Attack: 3
  Privilege Escalation: 3
  Credential Dumping: 3
  Command and Control: 3
  Credential Harvesting: 3
  Insider Threat: 3
  Cryptojacking: 3
  Zero-Day Exploit: 3
  Supply Chain Attack: 3
  Living-off-the-Land Attack: 3
  Brute Force Attack: 3
  SQL Injection: 3
  Backdoor Installation: 3
  Business Email Compromise: 3
  Data Leak: 3

=== Files saved to Google Drive ===
  /content/drive/MyDrive/irp_retraining/datasets/ir_playbooks_alpaca_rewritten.jsonl
  /content/drive/MyDrive/irp_retraining/datasets/ir_playbooks_synthetic_v2.jsonl

Download these files and place in your local project datasets/ folder.
